In [1]:
import os
from dotenv import load_dotenv

load_dotenv("../.env")

True

In [2]:
import dashscope

api_key = os.getenv("DASHSCOPE_API_KEY")
dashscope.base_http_api_url = "https://dashscope-intl.aliyuncs.com/api/v1"  # endpoint 

In [13]:
"""
=============================================================================
DashScope Video Generation — Complete API Reference
=============================================================================
Covers:
  - Text-to-Video (T2V)   →  wan2.6-t2v, wan2.2-t2v-plus, wan2.1-t2v-turbo
  - Image-to-Video (I2V)  →  wan2.6-i2v-flash, wan2.2-i2v-plus, wan2.1-i2v-turbo

Docs (last updated Feb 2026):
  T2V: https://www.alibabacloud.com/help/en/model-studio/text-to-video-api-reference
  I2V: https://www.alibabacloud.com/help/en/model-studio/image-to-video-api-reference

Key rules:
  - All video generation is ASYNCHRONOUS: you submit a task, get a task_id,
    then poll until it's done. Do NOT re-submit — poll the existing task.
  - The model, API key, and endpoint URL must all belong to the SAME region.
  - Video URLs in the result are only valid for 24 hours. Download immediately.
  - task_id is also valid for 24 hours, after which status returns UNKNOWN.
=============================================================================
"""

import time
import requests
from http import HTTPStatus
from dashscope import VideoSynthesis

# =============================================================================
# AVAILABLE MODELS (as of Feb 2026)
# =============================================================================
#
# TEXT-TO-VIDEO:
#   wan2.6-t2v           → Flagship. 1080P default, 2–15s, audio support.
#   wan2.6-t2v-us        → US region only. 5 or 10s fixed.
#   wan2.5-t2v-preview   → Good quality, 480P–1080P, 5 or 10s.
#   wan2.2-t2v-plus      → Solid quality, silent only, fixed 5s.
#   wan2.1-t2v-turbo     → Fast + cheap, silent only, fixed 5s, max 720P.
#   wan2.1-t2v-plus      → Silent, fixed 5s, 720P only.
#
# IMAGE-TO-VIDEO:
#   wan2.6-i2v-flash     → Flagship. 720P/1080P, 2–15s, audio support.
#   wan2.6-i2v           → Full version of above (slightly slower).
#   wan2.5-i2v-preview   → Good quality, 480P–1080P, 5 or 10s.
#   wan2.2-i2v-plus      → Silent only, fixed 5s, 480P or 1080P.
#   wan2.2-i2v-flash     → Fast silent, fixed 5s, 480P or 720P.
#   wan2.1-i2v-turbo     → Cheapest, 3–5s, 480P or 720P.
#   wan2.1-i2v-plus      → Silent, fixed 5s, 720P only.
#
# PRICING NOTE:
#   Cost = (unit price based on resolution) × (duration in seconds)
#   1080P > 720P > 480P. Audio videos cost more than silent.
#   Always check: https://www.alibabacloud.com/help/en/model-studio/models


# =============================================================================
# HELPER — Poll for task result
# =============================================================================

def wait_for_task(task_id: str, poll_interval: int = 15) -> dict | None:
    print(f"Polling task {task_id} every {poll_interval}s...")

    while True:
        status = VideoSynthesis.fetch(task_id, api_key=api_key)

        if status.status_code != HTTPStatus.OK:
            print(f"Poll request failed: {status.code} — {status.message}")
            return None

        task_status = status.output.task_status
        print(f"  Status: {task_status}")

        if task_status == "SUCCEEDED":
            return status
        elif task_status in ("FAILED", "CANCELED", "UNKNOWN"):
            # ← Print the actual error reason from the task output
            print(f"Task failed.")
            print(f"  code:    {status.output.get('code', 'N/A')}")
            print(f"  message: {status.output.get('message', 'N/A')}")
            return None

        time.sleep(poll_interval)


def download_video(url: str, filename: str) -> None:
    """
    Downloads the video from the result URL and saves it locally.

    IMPORTANT: The URL is only valid for 24 hours. Download immediately
    after the task succeeds, then store it in your own persistent storage
    (e.g. AWS S3, Alibaba OSS).
    """
    print(f"Downloading video to {filename}...")
    response = requests.get(url, stream=True)
    with open(filename, "wb") as f:
        for chunk in response.iter_content(chunk_size=8192):
            f.write(chunk)
    print(f"Saved: {filename}")


# =============================================================================
# TEXT-TO-VIDEO (T2V)
# =============================================================================

def text_to_video_basic():
    """
    Minimal T2V example using wan2.6-t2v.
    Generates a short video from a text prompt only.
    """
    rsp = VideoSynthesis.async_call(
        api_key=api_key,

        # ---------- Required ----------

        model="wan2.1-t2v-turbo",
        # ^ Model to use. See model list above.

        prompt=(
            "A red fox sits at a wooden desk inside a cozy cabin, "
            "writing in a journal by candlelight. Snow falls gently outside the window."
        ),
        # ^ Describe what you want in the video. Be descriptive.
        #   Supports English and Chinese. Max 1500 chars for wan2.6/2.5,
        #   max 800 chars for wan2.2/2.1.

        # ---------- Optional ----------

        size="1280*720",
        # ^ Resolution as "width*height". Must be an exact value, not a ratio.
        #   480P options: "832*480" (16:9), "480*832" (9:16), "624*624" (1:1)
        #   720P options: "1280*720", "720*1280", "960*960", "1088*832", "832*1088"
        #   1080P options: "1920*1080", "1080*1920", "1440*1440", "1632*1248", "1248*1632"
        #   Default for wan2.6-t2v is "1920*1080". Higher res = higher cost.

        duration=5,
        # ^ Video length in seconds.
        #   wan2.6-t2v: integer 2–15. Default: 5.
        #   wan2.5-t2v-preview: 5 or 10.
        #   wan2.2 and wan2.1 series: fixed at 5 (cannot be changed).

        prompt_extend=True,
        # ^ If True, an LLM rewrites your prompt before generation.
        #   Significantly improves quality for short/vague prompts.
        #   Adds a few seconds of processing time. Default: True.

        watermark=False,
        # ^ If True, adds "AI Generated" text watermark to bottom-right corner.
        #   Default: False.

        seed=42,
        # ^ Integer 0–2147483647. Set a fixed seed for more reproducible results.
        #   Due to model randomness, same seed does not guarantee identical output.
        #   Omit (or set to None) to use a random seed each time.
    )

    if rsp.status_code != HTTPStatus.OK:
        print(f"Task creation failed: {rsp.code} — {rsp.message}")
        return

    task_id = rsp.output.task_id
    print(f"Task created. task_id: {task_id}")

    result = wait_for_task(task_id)
    if result:
        video_url = result.output.video_url
        print(f"Video ready: {video_url}")
        download_video(video_url, "t2v_basic.mp4")


def text_to_video_with_audio():
    """
    T2V with custom audio using wan2.6-t2v.
    wan2.6 and wan2.5 models support audio; older models generate silent videos.
    """
    rsp = VideoSynthesis.async_call(
        api_key=api_key,
        model="wan2.6-t2v",

        prompt=(
            "A street musician plays violin under a glowing lantern at dusk. "
            "Passersby slow down to listen. The camera slowly circles the musician."
        ),

        audio_url="https://your-public-audio-url.com/music.mp3",
        # ^ URL of a custom audio file (WAV or MP3).
        #   The model syncs the video to this audio.
        #   Duration: 3–30s. File size: max 15 MB.
        #   If audio is longer than the video, it's truncated.
        #   If audio is shorter, the remaining video is silent.
        #   Omit this field → model auto-generates background audio from the prompt.

        size="832*480",
        duration=5,
        prompt_extend=True,
        watermark=False,
    )

    if rsp.status_code != HTTPStatus.OK:
        print(f"Failed: {rsp.code} — {rsp.message}")
        return

    result = wait_for_task(rsp.output.task_id)
    if result:
        download_video(result.output.video_url, "t2v_with_audio.mp4")


def text_to_video_multishot():
    """
    T2V with multi-shot narrative (wan2.6 only).
    Generates a video that cuts between multiple shots, like a short film.
    Requires prompt_extend=True to take effect.
    """
    rsp = VideoSynthesis.async_call(
        api_key=api_key,
        model="wan2.6-t2v",

        prompt=(
            "A detective enters a rainy city alley. Cut to close-up of a clue on the ground. "
            "Wide shot of the skyline at night. The detective looks up at a lit window."
        ),

        size="832*480",
        duration=5,

        prompt_extend=True,
        # ^ REQUIRED for shot_type to work. Must be True.

        shot_type="multi",
        # ^ "single" (default): one continuous shot throughout the video.
        #   "multi": multiple shots with cuts, like a narrative film.
        #   Note: shot_type takes priority over the prompt. If you set "single"
        #   but your prompt says "cut to...", the model still uses one shot.
        #   Only supported by wan2.6 series models.

        watermark=False,
    )

    if rsp.status_code != HTTPStatus.OK:
        print(f"Failed: {rsp.code} — {rsp.message}")
        return

    result = wait_for_task(rsp.output.task_id)
    if result:
        download_video(result.output.video_url, "t2v_multishot.mp4")


def text_to_video_with_negative_prompt():
    """
    T2V with a negative prompt to exclude unwanted elements.
    Useful for older models (wan2.2, wan2.1) or when you need strict control.
    """
    rsp = VideoSynthesis.async_call(
        api_key=api_key,
        model="wan2.2-t2v-plus",
        # ^ wan2.2 generates silent videos (no audio support).
        #   Fixed at 5 seconds duration.

        prompt="A kitten plays in a sunlit garden",

        negative_prompt="flowers, blurry, low quality, distorted limbs",
        # ^ Describe what you DON'T want in the video.
        #   Max 500 characters. Any excess is automatically truncated.
        #   Common use: exclude quality artifacts, specific objects, styles.

        size="832*480",
        # ^ 480P is cheapest. Good for testing or lower-budget generation.

        prompt_extend=True,
        watermark=False,
    )

    if rsp.status_code != HTTPStatus.OK:
        print(f"Failed: {rsp.code} — {rsp.message}")
        return

    result = wait_for_task(rsp.output.task_id)
    if result:
        download_video(result.output.video_url, "t2v_negative_prompt.mp4")


# =============================================================================
# IMAGE-TO-VIDEO (I2V)
# =============================================================================

def image_to_video_basic():
    """
    Minimal I2V example using wan2.6-i2v-flash.
    Animates a first-frame image according to a text prompt.
    The output video's aspect ratio matches the input image's aspect ratio.
    """
    rsp = VideoSynthesis.async_call(
        api_key=api_key,

        # ---------- Required ----------

        model="wan2.2-i2v-flash",
        # ^ Flagship I2V model. Supports audio, 720P/1080P, 2–15s.

        img_url="https://cdn.translate.alibaba.com/r/wanx-demo-1.png",
        # ^ URL of the first-frame image. This becomes the opening frame of the video.
        #   Supported formats: JPEG, JPG, PNG (no alpha), BMP, WEBP.
        #   Resolution: width and height must each be between 360 and 2000 pixels.
        #   File size: max 10 MB.
        #   Can also pass a Base64 string: "data:image/png;base64,<base64_data>"
        #   The output video aspect ratio follows the image's aspect ratio.

        # ---------- Optional ----------

        prompt="The man stretches, yawns, then slowly walks toward the camera.",
        # ^ Describes how the image should be animated.
        #   For I2V, the prompt is technically optional (the model can infer motion),
        #   but providing one gives you much more control over the animation.
        #   Max 1500 chars for wan2.6/2.5, 800 chars for wan2.2/2.1.

        resolution="480P",
        # ^ Resolution tier: "480P", "720P", or "1080P".
        #   NOTE: I2V uses resolution TIERS, not exact pixel dimensions.
        #   The model picks exact pixel dimensions based on the input image's aspect ratio.
        #   Available tiers depend on the model (see model list above).
        #   Default for wan2.6-i2v-flash: "1080P".

        duration=5,
        # ^ Video length in seconds.
        #   wan2.6-i2v-flash: integer 2–15. Default: 5.
        #   wan2.5-i2v-preview: 5 or 10.
        #   wan2.2 series and wan2.1-i2v-plus: fixed at 5 (cannot be changed).
        #   wan2.1-i2v-turbo: 3, 4, or 5.

        prompt_extend=True,
        # ^ Rewrites prompt with LLM for richer results. Default: True.

        watermark=False,
        seed=42,
    )

    if rsp.status_code != HTTPStatus.OK:
        print(f"Task creation failed: {rsp.code} — {rsp.message}")
        return

    task_id = rsp.output.task_id
    print(f"Task created. task_id: {task_id}")

    result = wait_for_task(task_id)
    if result:
        video_url = result.output.video_url
        print(f"Video ready: {video_url}")
        download_video(video_url, "i2v_basic.mp4")


def image_to_video_with_audio():
    """
    I2V with custom audio file synced to the video (wan2.6/wan2.5 only).
    """
    rsp = VideoSynthesis.async_call(
        api_key=api_key,
        model="wan2.6-i2v-flash",

        img_url="https://your-image-url.com/scene.jpg",

        prompt=(
            "A graffiti artist steps back to admire their mural. "
            "The mural starts glowing with neon colors. The artist smiles."
        ),

        audio_url="https://your-public-audio-url.com/soundtrack.mp3",
        # ^ Optional: custom audio to sync with the video.
        #   Same constraints as T2V: WAV/MP3, 3–30s, max 15 MB.
        #   Omit → model auto-generates audio from the video content.

        resolution="720P",
        duration=10,
        prompt_extend=True,
        watermark=False,
    )

    if rsp.status_code != HTTPStatus.OK:
        print(f"Failed: {rsp.code} — {rsp.message}")
        return

    result = wait_for_task(rsp.output.task_id)
    if result:
        download_video(result.output.video_url, "i2v_with_audio.mp4")


def image_to_video_silent():
    """
    I2V generating a silent video.
    - For wan2.2/wan2.1 models: silent by default, no extra param needed.
    - For wan2.6-i2v-flash: must explicitly set audio=False.
    """
    rsp = VideoSynthesis.async_call(
        api_key=api_key,
        model="wan2.6-i2v-flash",

        img_url="https://cdn.translate.alibaba.com/r/wanx-demo-1.png",
        prompt="A cat walks slowly through tall grass in golden hour light.",

        resolution="720P",
        duration=5,
        prompt_extend=True,

        audio=False,
        # ^ wan2.6-i2v-flash only: explicitly disables audio generation.
        #   IMPORTANT: If audio=False, the video is silent even if you pass audio_url.
        #   audio=False uses the silent-video pricing tier (cheaper).
        #   For wan2.2/wan2.1 models, silence is the default — no need to set this.

        watermark=False,
    )

    if rsp.status_code != HTTPStatus.OK:
        print(f"Failed: {rsp.code} — {rsp.message}")
        return

    result = wait_for_task(rsp.output.task_id)
    if result:
        download_video(result.output.video_url, "i2v_silent.mp4")


def image_to_video_multishot():
    """
    I2V with multi-shot narrative (wan2.6 models only).
    The first frame becomes the opening image, and the video cuts between shots.
    """
    rsp = VideoSynthesis.async_call(
        api_key=api_key,
        model="wan2.6-i2v-flash",

        img_url="https://your-image-url.com/city_alley.jpg",

        prompt=(
            "A boy made of spray paint comes to life on a concrete wall. "
            "He raps at high speed. The camera cuts to a crowd watching. "
            "Wide shot of the street at night, neon lights everywhere."
        ),

        audio_url="https://your-audio-url.com/rap.mp3",
        # ^ Optional audio to sync with the multi-shot video.

        resolution="720P",
        duration=10,

        prompt_extend=True,
        # ^ Must be True for shot_type to work.

        shot_type="multi",
        # ^ Enables multi-shot narrative: the video cuts between different shots.
        #   "single" = one continuous shot (default).
        #   "multi"  = multiple shots with cuts.
        #   Only supported on wan2.6 series.

        watermark=False,
    )

    if rsp.status_code != HTTPStatus.OK:
        print(f"Failed: {rsp.code} — {rsp.message}")
        return

    result = wait_for_task(rsp.output.task_id)
    if result:
        download_video(result.output.video_url, "i2v_multishot.mp4")


def image_to_video_with_negative_prompt():
    """
    I2V with negative prompt to exclude unwanted elements.
    Most useful for older models (wan2.2, wan2.1).
    """
    rsp = VideoSynthesis.async_call(
        api_key=api_key,
        model="wan2.2-i2v-plus",
        # ^ wan2.2-i2v-plus: silent, fixed 5s, 480P or 1080P.

        img_url="https://cdn.translate.alibaba.com/r/wanx-demo-1.png",
        prompt="A cat chases a butterfly across a meadow.",

        negative_prompt="flowers, blurry, low resolution, distorted body",
        # ^ Describe what to EXCLUDE. Max 500 characters.

        resolution="480P",
        prompt_extend=True,
        watermark=False,
    )

    if rsp.status_code != HTTPStatus.OK:
        print(f"Failed: {rsp.code} — {rsp.message}")
        return

    result = wait_for_task(rsp.output.task_id)
    if result:
        download_video(result.output.video_url, "i2v_negative_prompt.mp4")


# =============================================================================
# USING BASE64 IMAGE INPUT (for I2V when you have a local file)
# =============================================================================

def image_to_video_from_local_file(image_path: str):
    """
    I2V using a local image file encoded as Base64.
    Useful when your image isn't hosted at a public URL.
    """
    import base64

    with open(image_path, "rb") as f:
        image_data = f.read()

    # Encode to Base64 and format as a data URL
    b64_string = base64.b64encode(image_data).decode("utf-8")
    img_url = f"data:image/png;base64,{b64_string}"
    # ^ Format: "data:{MIME_type};base64,{base64_data}"
    # Supported MIME types: image/jpeg, image/png, image/bmp, image/webp

    rsp = VideoSynthesis.async_call(
        api_key=api_key,
        model="wan2.2-i2v-flash",
        img_url=img_url,   # Base64 string instead of public URL
        prompt="The man stretches, yawns, then slowly walks toward the camera.",
        resolution="480P",
        duration=5,
        prompt_extend=False,
        watermark=False,
    )

    if rsp.status_code != HTTPStatus.OK:
        print(f"Failed: {rsp.code} — {rsp.message}")
        return

    result = wait_for_task(rsp.output.task_id)
    if result:
        download_video(result.output.video_url, "i2v_from_local.mp4")


# =============================================================================
# RESPONSE STRUCTURE — What you get back
# =============================================================================
#
# After async_call():
# {
#   "output": {
#     "task_id":     "38513c71-...",   ← Use this to poll for the result
#     "task_status": "PENDING"         ← Initial status
#   },
#   "request_id":    "abbf7aa3-..."    ← For support/debugging
# }
#
# After the task SUCCEEDS (from fetch() or wait()):
# {
#   "output": {
#     "task_id":        "38513c71-...",
#     "task_status":    "SUCCEEDED",
#     "video_url":      "https://dashscope-result-bj.oss-accelerate.aliyuncs.com/xxx.mp4?Expires=xxx",
#     "submit_time":    "2025-10-23 11:47:23.879",
#     "scheduled_time": "2025-10-23 11:47:34.351",
#     "end_time":       "2025-10-23 11:52:35.323",
#     "orig_prompt":    "The original prompt text after rewriting (if prompt_extend=True)"
#   }
# }
#
# IMPORTANT: video_url expires after 24 hours. Download and store it immediately.


# =============================================================================
# QUICK REFERENCE — T2V vs I2V parameter differences
# =============================================================================
#
# Parameter         T2V         I2V         Notes
# -----------------------------------------------------------------------
# model             Required    Required    Different model names
# prompt            Required    Optional    I2V can infer motion without prompt
# img_url           N/A         Required    First frame image for I2V
# negative_prompt   Optional    Optional    Both support it
# audio_url         Optional    Optional    wan2.6 / wan2.5 only
# size              Optional    N/A         T2V uses exact "WxH" pixel dimensions
# resolution        N/A         Optional    I2V uses tier: "480P"/"720P"/"1080P"
# duration          Optional    Optional    Varies by model
# prompt_extend     Optional    Optional    Default True for both
# shot_type         Optional    Optional    wan2.6 only, requires prompt_extend=True
# audio             N/A         Optional    wan2.6-i2v-flash only (True/False)
# watermark         Optional    Optional    Default False for both
# seed              Optional    Optional    0–2147483647


# =============================================================================
# RUN EXAMPLES
# =============================================================================

if __name__ == "__main__":
    # Uncomment whichever example you want to run:

    # --- Text-to-Video ---
    # text_to_video_basic()
    # text_to_video_with_audio()
    # text_to_video_multishot()
    # text_to_video_with_negative_prompt()

    # --- Image-to-Video ---
    # image_to_video_basic()
    # image_to_video_with_audio()
    # image_to_video_silent()
    # image_to_video_multishot()
    # image_to_video_with_negative_prompt()
    image_to_video_from_local_file("../legoman.png")

Polling task a91fe979-0eed-421d-b9b7-7789d21af220 every 15s...
  Status: RUNNING
  Status: RUNNING
  Status: SUCCEEDED
Saved: i2v_from_local.mp4


In [ ]:
# TODO: I want to test the open source 2.2 model. Maybe I am able to run this locally? Also it seems like the 2.2 model is very good and maybe I therefore dont need 2.6. But the idea could be that the video pre generates some clips, and the clips is always 30 seconds in front of the users intent. Ideally this should be around 5 - 10 seconds. Lets go!